In [1]:
"""
Tumor Segmentation - Attention U-Net with EfficientNetB3 Encoder
====================================================================
Trains a segmentation model on the pseudo-masks generated by mask_generator.py.
Classes covered: glioma, meningioma, pituitary (notumor is skipped - nothing
to segment there).

IMPORTANT: masks were generated at 256x256 (mask_generator.py's default), so
this script also uses IMG_SIZE=256 to match - NOT 300 like the classifier.

Expected folder structure (already created by earlier scripts):
    dataset/Training/{glioma,meningioma,pituitary}/*.jpg              <- original images
    dataset/Masks/Training/{glioma,meningioma,pituitary}/*_mask.png   <- generated masks
    dataset/Testing/... (same structure)

Run this from inside python-backend/segmentation/, e.g.:
    python train_unet.py
"""

"\nTumor Segmentation - Attention U-Net with EfficientNetB3 Encoder\n====================================================================\nTrains a segmentation model on the pseudo-masks generated by mask_generator.py.\nClasses covered: glioma, meningioma, pituitary (notumor is skipped - nothing\nto segment there).\n\nIMPORTANT: masks were generated at 256x256 (mask_generator.py's default), so\nthis script also uses IMG_SIZE=256 to match - NOT 300 like the classifier.\n\nExpected folder structure (already created by earlier scripts):\n    dataset/Training/{glioma,meningioma,pituitary}/*.jpg              <- original images\n    dataset/Masks/Training/{glioma,meningioma,pituitary}/*_mask.png   <- generated masks\n    dataset/Testing/... (same structure)\n\nRun this from inside python-backend/segmentation/, e.g.:\n    python train_unet.py\n"

In [2]:
import os
import glob
import numpy as np
import cv2
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB3
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.layers import (
    Conv2D, Conv2DTranspose, BatchNormalization, Activation,
    Concatenate, multiply, add, UpSampling2D
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
import matplotlib.pyplot as plt

In [3]:
# ----------------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------------
IMG_SIZE = 256          # MUST match mask_generator.py's output size
BATCH_SIZE = 1          # segmentation is more memory-hungry than classification, lower default
CLASSES = ["glioma", "meningioma", "pituitary"]   # notumor excluded - nothing to segment

In [4]:
TRAIN_IMG_ROOT = "../dataset/Segmentation/Training/images"
TRAIN_MASK_ROOT = "../dataset/Segmentation/Training/masks"
TEST_IMG_ROOT = "../dataset/Segmentation/Testing/images"
TEST_MASK_ROOT = "../dataset/Segmentation/Testing/masks"

In [5]:
EPOCHS = 40
LEARNING_RATE = 1e-4
VAL_SPLIT = 0.15

In [6]:
MODEL_WEIGHTS_OUT = "../models/unet_segmentation.weights.h5"
# NOTE: we save WEIGHTS ONLY (not the full model via model.save()), same lesson
# learned from the classification model - saving/loading full .h5 models hit a
# Keras version bug earlier. Saving weights + rebuilding the architecture in
# code (as build_model() does below) sidesteps that entirely and is more robust.

In [7]:
# ----------------------------------------------------------------------------
# STEP 1: Collect image/mask file pairs
# ----------------------------------------------------------------------------
def collect_pairs(img_root, mask_root):
    """
    BRISC structure: img_root and mask_root are flat folders (no per-class
    subfolders). Each image (e.g. brisc2025_train_00001_gl_ax_t1.jpg) has a
    matching mask with the SAME basename but .png extension
    (brisc2025_train_00001_gl_ax_t1.png) - no '_mask' suffix, unlike our old
    pseudo-mask setup.
    """
    image_paths, mask_paths = [], []

    if not os.path.exists(img_root) or not os.path.exists(mask_root):
        print(f"[WARN] Missing folder: {img_root} or {mask_root}")
        return image_paths, mask_paths

    for img_path in sorted(glob.glob(os.path.join(img_root, "*"))):
        if not img_path.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        stem = os.path.splitext(os.path.basename(img_path))[0]
        mask_path = os.path.join(mask_root, f"{stem}.png")
        if os.path.exists(mask_path):
            image_paths.append(img_path)
            mask_paths.append(mask_path)
        else:
            print(f"[WARN] No matching mask for {os.path.basename(img_path)}")

    return image_paths, mask_paths

In [8]:
# ----------------------------------------------------------------------------
# STEP 2: Paired data generator (custom Sequence - keeps image+mask in sync)
# ----------------------------------------------------------------------------
class SegmentationDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, image_paths, mask_paths, batch_size, img_size, augment=False, shuffle=True):
        self.image_paths = image_paths
        self.mask_paths = mask_paths
        self.batch_size = batch_size
        self.img_size = img_size
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(self.image_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_images = np.zeros((len(batch_indices), self.img_size, self.img_size, 3), dtype=np.float32)
        batch_masks = np.zeros((len(batch_indices), self.img_size, self.img_size, 1), dtype=np.float32)

        for i, data_idx in enumerate(batch_indices):
            img = cv2.imread(self.image_paths[data_idx], cv2.IMREAD_COLOR)
            img = cv2.resize(img, (self.img_size, self.img_size))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            mask = cv2.imread(self.mask_paths[data_idx], cv2.IMREAD_GRAYSCALE)
            mask = cv2.resize(mask, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)
            mask = (mask > 127).astype(np.float32)

            if self.augment:
                img, mask = self._augment(img, mask)

            batch_images[i] = preprocess_input(img.astype(np.float32))
            batch_masks[i] = mask[..., np.newaxis]

        return batch_images, batch_masks

    def _augment(self, img, mask):
        """Applies the SAME random transform to both image and mask - critical,
        otherwise they'd desync and the model would learn garbage."""
        if np.random.rand() < 0.5:   # horizontal flip
            img = cv2.flip(img, 1)
            mask = cv2.flip(mask, 1)
        if np.random.rand() < 0.5:   # vertical flip
            img = cv2.flip(img, 0)
            mask = cv2.flip(mask, 0)
        if np.random.rand() < 0.5:   # small rotation
            angle = np.random.uniform(-15, 15)
            h, w = img.shape[:2]
            M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
            img = cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_CONSTANT)
            mask = cv2.warpAffine(mask, M, (w, h), borderMode=cv2.BORDER_CONSTANT)
        return img, mask

In [9]:
# ----------------------------------------------------------------------------
# STEP 3: Model - Attention U-Net with EfficientNetB3 encoder
# ----------------------------------------------------------------------------
def get_skip_layer(base_model, name_substring):
    """Finds a layer by substring match instead of exact name - more robust
    across slightly different Keras/TF versions where exact layer names can vary."""
    matches = [l for l in base_model.layers if name_substring in l.name]
    if not matches:
        available = [l.name for l in base_model.layers if "activation" in l.name][:20]
        raise ValueError(
            f"No layer found containing '{name_substring}'. "
            f"Some available activation layer names:\n{available}"
        )
    return matches[-1].output   # last match (deepest occurrence of that block)

In [10]:
def attention_gate(x, gating, inter_channels):
    """
    Attention gate: lets the decoder learn to suppress irrelevant encoder
    (skip connection) regions and focus on tumor-relevant areas only.
    """
    theta_x = Conv2D(inter_channels, 1, strides=1, padding="same")(x)
    phi_g = Conv2D(inter_channels, 1, strides=1, padding="same")(gating)

    if phi_g.shape[1] != theta_x.shape[1]:
        phi_g = UpSampling2D(
            size=(theta_x.shape[1] // phi_g.shape[1], theta_x.shape[2] // phi_g.shape[2]),
            interpolation="bilinear"
        )(phi_g)

    concat = add([theta_x, phi_g])
    act = Activation("relu")(concat)
    psi = Conv2D(1, 1, strides=1, padding="same", activation="sigmoid")(act)
    return multiply([x, psi])

In [11]:
def conv_block(x, filters):
    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    x = Conv2D(filters, 3, padding="same")(x)
    x = BatchNormalization()(x)
    x = Activation("relu")(x)
    return x

In [12]:
def decoder_block(x, skip, filters):
    x = Conv2DTranspose(filters, 2, strides=2, padding="same")(x)

    # Dynamically resize skip to match x's spatial size before attention/concat.
    # We don't hardcode expected resolutions here - EfficientNet layer resolutions
    # can differ slightly across TF/Keras versions (learned this the hard way with
    # top_conv earlier), so we just always force-match at runtime instead of assuming.
    if skip.shape[1] != x.shape[1] or skip.shape[2] != x.shape[2]:
        skip = tf.keras.layers.Resizing(
            x.shape[1], x.shape[2], interpolation="bilinear"
        )(skip)

    attended_skip = attention_gate(skip, x, filters // 2)
    x = Concatenate()([x, attended_skip])
    x = conv_block(x, filters)
    return x

In [13]:
def build_model():
    base_model = EfficientNetB3(
        include_top=False, weights="imagenet", input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    # Skip connections at increasing resolution (shallow -> deep)
    s1 = get_skip_layer(base_model, "stem_activation")              # ~128x128
    s2 = get_skip_layer(base_model, "block2a_expand_activation")    # ~64x64
    s3 = get_skip_layer(base_model, "block3a_expand_activation")    # ~32x32
    s4 = get_skip_layer(base_model, "block4a_expand_activation")    # ~16x16
    bottleneck = base_model.output                                  # ~8x8

    d1 = decoder_block(bottleneck, s4, 256)   # -> 16x16
    d2 = decoder_block(d1, s3, 128)           # -> 32x32
    d3 = decoder_block(d2, s2, 64)            # -> 64x64
    d4 = decoder_block(d3, s1, 32)            # -> 128x128

    # Final upsample back to full input resolution
    x = Conv2DTranspose(16, 2, strides=2, padding="same")(d4)  # -> 256x256
    x = conv_block(x, 16)
    outputs = Conv2D(1, 1, activation="sigmoid")(x)

    return Model(inputs=base_model.input, outputs=outputs)

In [14]:
# ----------------------------------------------------------------------------
# STEP 4: Loss + Metrics - Dice + BCE combo (standard for segmentation)
# ----------------------------------------------------------------------------
def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(y_pred)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

In [15]:
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coefficient(y_true, y_pred)

In [16]:
def combined_bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)

In [17]:
def iou_metric(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.keras.backend.flatten(y_true)
    y_pred_f = tf.keras.backend.flatten(tf.cast(y_pred > 0.5, tf.float32))
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

In [18]:
# ----------------------------------------------------------------------------
# STEP 5: MAIN TRAINING
# ----------------------------------------------------------------------------
def main():
    print("Collecting training image/mask pairs...")
    all_img_paths, all_mask_paths = collect_pairs(TRAIN_IMG_ROOT, TRAIN_MASK_ROOT)
    print(f"Found {len(all_img_paths)} training pairs (BRISC segmentation set covers: {CLASSES})")

    if len(all_img_paths) == 0:
        print("No training pairs found. Check that mask_generator.py has been run "
              "and paths in this script match your folder structure.")
        return

    # Manual train/val split (shuffled, fixed seed for reproducibility)
    rng = np.random.RandomState(42)
    idx = rng.permutation(len(all_img_paths))
    split_at = int(len(idx) * (1 - VAL_SPLIT))
    train_idx, val_idx = idx[:split_at], idx[split_at:]

    train_imgs = [all_img_paths[i] for i in train_idx]
    train_masks = [all_mask_paths[i] for i in train_idx]
    val_imgs = [all_img_paths[i] for i in val_idx]
    val_masks = [all_mask_paths[i] for i in val_idx]

    print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")

    train_gen = SegmentationDataGenerator(train_imgs, train_masks, BATCH_SIZE, IMG_SIZE, augment=True, shuffle=True)
    val_gen = SegmentationDataGenerator(val_imgs, val_masks, BATCH_SIZE, IMG_SIZE, augment=False, shuffle=False)

    print("Building Attention U-Net model...")
    model = build_model()
    model.summary()

    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss=combined_bce_dice_loss,
        metrics=[dice_coefficient, iou_metric],
    )

    class PrintMetricsCallback(tf.keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            logs = logs or {}
            print(f"\n--- Epoch {epoch + 1} Summary ---")
            print(f"Loss: {logs.get('loss', 0):.4f} | Dice: {logs.get('dice_coefficient', 0):.4f} | IoU: {logs.get('iou_metric', 0):.4f}")
            print(f"Val Loss: {logs.get('val_loss', 0):.4f} | Val Dice: {logs.get('val_dice_coefficient', 0):.4f} | Val IoU: {logs.get('val_iou_metric', 0):.4f}")
            print("---------------------------\n")

    callbacks = [
        PrintMetricsCallback(),
        ModelCheckpoint(
            MODEL_WEIGHTS_OUT, monitor="val_dice_coefficient", mode="max",
            save_best_only=True, save_weights_only=True
        ),
        EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, min_lr=1e-7),
        CSVLogger("unet_training_log.csv"),
    ]

    print("Starting training...")
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=EPOCHS,
        callbacks=callbacks,
    )

    # ------------------------------------------------------------------------
    # Plot training curves
    # ------------------------------------------------------------------------
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(history.history["loss"], label="Train Loss")
    axes[0].plot(history.history["val_loss"], label="Val Loss")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(history.history["dice_coefficient"], label="Train Dice")
    axes[1].plot(history.history["val_dice_coefficient"], label="Val Dice")
    axes[1].set_title("Dice Coefficient")
    axes[1].legend()

    axes[2].plot(history.history["iou_metric"], label="Train IoU")
    axes[2].plot(history.history["val_iou_metric"], label="Val IoU")
    axes[2].set_title("IoU")
    axes[2].legend()

    plt.tight_layout()
    plt.savefig("unet_training_curves.png", dpi=150)
    plt.close()

    print(f"\nDone. Best weights saved to {MODEL_WEIGHTS_OUT}")
    print("Training curves saved to unet_training_curves.png")

In [ ]:
if __name__ == "__main__":
    main()